# Notebook 05: Embedding Preparation

## Introduction

This notebook focuses on preparing and generating text embeddings from the final metadata dataset.

The embeddings represent the textual information associated with each image, including its captions and object categories.

The generated embeddings will be saved and organized for future retrieval operations.

This notebook is part of Phase 1: Data Exploration and Preprocessing.

## Step 2: Load Final Metadata

This step loads the final metadata dataset generated in Notebook 04.

The metadata contains image information, original and cleaned captions, and object annotations.

It will serve as the input dataset for the embedding preparation process.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\PC\Desktop\Metadata_RAG_START")
DATA_DIR = PROJECT_DIR / "data"
EMBEDDINGS_DIR = PROJECT_DIR / "embeddings"

METADATA_PATH = DATA_DIR / "metadata.csv"

metadata_df = pd.read_csv(METADATA_PATH)

print("Metadata loaded successfully!")
print("Dataset shape:", metadata_df.shape)

display(metadata_df.head())

Metadata loaded successfully!
Dataset shape: (10099, 14)


,image_id,file_name,dataset,processed_path,width,height,original_mode,processed_mode,original_captions,cleaned_captions,caption_count,objects,object_count,object_categories
0,36,000000000036.jpg,training,C:\Users\PC\Desktop\Metadata_RAG_START\data\pr...,481,640,RGB,RGB,['Woman in swim suit holding parasol on sunny ...,['Woman in swim suit holding parasol on sunny ...,5,"[{'category': 'umbrella', 'category_id': 28, '...",2,"['person', 'umbrella']"
1,73,000000000073.jpg,training,C:\Users\PC\Desktop\Metadata_RAG_START\data\pr...,565,640,RGB,RGB,['A motorcycle parked in a parking space next ...,['A motorcycle parked in a parking space next ...,5,"[{'category': 'motorcycle', 'category_id': 4, ...",2,['motorcycle']
2,89,000000000089.jpg,training,C:\Users\PC\Desktop\Metadata_RAG_START\data\pr...,640,480,RGB,RGB,['An oven with a stove on top of it in a kitch...,['An oven with a stove on top of it in a kitch...,5,"[{'category': 'knife', 'category_id': 49, 'bbo...",10,"['book', 'knife', 'microwave', 'oven']"
3,109,000000000109.jpg,training,C:\Users\PC\Desktop\Metadata_RAG_START\data\pr...,640,416,RGB,RGB,['A skate park next to a body of water and gre...,['A skate park next to a body of water and gre...,5,"[{'category': 'dog', 'category_id': 18, 'bbox'...",8,"['bench', 'dog', 'person']"
4,196,000000000196.jpg,training,C:\Users\PC\Desktop\Metadata_RAG_START\data\pr...,640,480,RGB,RGB,['This table is filled with a variety of diffe...,['This table is filled with a variety of diffe...,5,"[{'category': 'spoon', 'category_id': 50, 'bbo...",42,"['bowl', 'broccoli', 'carrot', 'dining table',..."


## Step 3: Prepare Text for Embedding

This step prepares a textual representation for each image.

The text combines the cleaned captions and object categories associated with each image.

This representation provides descriptive information about the image and will be used as input for the embedding model.

### 3.1 Create Text Descriptions

For each image, the five cleaned captions and the detected object categories are combined into a single text description.

This creates one textual document per image while preserving its image ID.

In [2]:
import ast

def prepare_image_text(row):
    captions = row["cleaned_captions"]
    categories = row["object_categories"]

    # Convert string representations of lists into Python lists
    if isinstance(captions, str):
        captions = ast.literal_eval(captions)

    if isinstance(categories, str):
        categories = ast.literal_eval(categories)

    captions_text = " ".join(captions)
    objects_text = ", ".join(categories)

    return (
        f"Captions: {captions_text} "
        f"Objects: {objects_text}"
    )


metadata_df["embedding_text"] = metadata_df.apply(
    prepare_image_text,
    axis=1
)

print("Text preparation completed!")
print("Total text descriptions:", len(metadata_df))
display(metadata_df[["image_id", "embedding_text"]].head())

Text preparation completed!
Total text descriptions: 10099


,image_id,embedding_text
0,36,Captions: Woman in swim suit holding parasol o...
1,73,Captions: A motorcycle parked in a parking spa...
2,89,Captions: An oven with a stove on top of it in...
3,109,Captions: A skate park next to a body of water...
4,196,Captions: This table is filled with a variety ...


### 3.2 Verify Prepared Text

This step checks the generated textual descriptions.

It verifies that no descriptions are missing or empty and displays an example for inspection.

In [4]:
print("Total descriptions:", len(metadata_df))

print(
    "Missing descriptions:",
    metadata_df["embedding_text"].isna().sum()
)

print(
    "Empty descriptions:",
    (metadata_df["embedding_text"].str.strip() == "").sum()
)

print("\nExample description:")
print(metadata_df.loc[0, "embedding_text"])

Total descriptions: 10099
Missing descriptions: 0
Empty descriptions: 0

Example description:
Captions: Woman in swim suit holding parasol on sunny day. A woman posing for the camera, holding a pink, open umbrella and wearing a bright, floral, ruched bathing suit, by a life guard stand with lake, green trees, and a blue sky with a few clouds behind. A woman in a floral swimsuit holds a pink umbrella. A woman with an umbrella near the sea A girl in a bathing suit with a pink umbrella. Objects: person, umbrella


## Step 4: Load the Embedding Model

This step prepares the Sentence Transformers library and loads a pre-trained text embedding model.

The selected model converts textual descriptions into numerical vector representations.

### 4.1 Install Sentence Transformers

The Sentence Transformers library is required to load the selected pre-trained embedding model.

In [5]:
%pip install -U sentence-transformers

   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ------------- -------------------------- 262.1/760.7 kB ? eta -:--:--
   ------------- -------------------------- 262.1/760.7 kB ? eta -:--:--
   -------------------------- ----------- 524.3/760.7 kB 885.6 kB/s eta 0:00:01
   ---------------------------------------- 760.7/760.7 kB 968.1 kB/s  0:00:01
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/857.0 kB ? eta -:--:--
   ------------------------ --------------- 524.3/857.0 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 857.0/857.0 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.9 MB ? eta -:--:--
   -------- -------------------------------

### 4.2 Load the Pre-trained Model

The all-MiniLM-L6-v2 model is loaded to generate 384-dimensional embeddings from the prepared image descriptions.

In [25]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully!")
print("Model name:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Model name: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\PC\AppData\Local\Temp\ipykernel_20176\2592934743.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## Step 5: Generate Text Embeddings

This step converts the prepared textual description of each image into a numerical embedding using the pre-trained Sentence Transformers model.

Each image will be represented by a 384-dimensional vector. The image IDs will be preserved to maintain the association between each embedding and its corresponding image.

In [7]:
# Generate one embedding for each image description

texts = metadata_df["embedding_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding generation completed!")
print("Number of embeddings:", embeddings.shape[0])
print("Embedding dimension:", embeddings.shape[1])
print("Total images:", len(metadata_df))

Batches:   0%|          | 0/316 [00:00<?, ?it/s]

Embedding generation completed!
Number of embeddings: 10099
Embedding dimension: 384
Total images: 10099


## Step 6: Save the Generated Embeddings

This step saves the generated embeddings to disk for future use.

The embeddings are stored in NumPy format, while a separate CSV file preserves the association between each image ID and its corresponding vector index.

This ensures that the embeddings can be reloaded without generating them again.

In [8]:
import numpy as np
import pandas as pd

# Create the embeddings directory if it does not exist
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Save the numerical embeddings
EMBEDDINGS_PATH = EMBEDDINGS_DIR / "image_embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)

# Save the mapping between image IDs and embedding indices
embedding_index_df = metadata_df[["image_id", "file_name", "dataset"]].copy()
embedding_index_df["embedding_index"] = range(len(embedding_index_df))

INDEX_PATH = EMBEDDINGS_DIR / "embedding_index.csv"
embedding_index_df.to_csv(INDEX_PATH, index=False)

print("Embeddings saved successfully!")
print("Embeddings file:", EMBEDDINGS_PATH)
print("Index file:", INDEX_PATH)
print("Embedding shape:", np.load(EMBEDDINGS_PATH, mmap_mode="r").shape)
print("Index records:", len(embedding_index_df))

Embeddings saved successfully!
Embeddings file: C:\Users\PC\Desktop\Metadata_RAG_START\embeddings\image_embeddings.npy
Index file: C:\Users\PC\Desktop\Metadata_RAG_START\embeddings\embedding_index.csv
Embedding shape: (10099, 384)
Index records: 10099


## Step 7: Verify the Saved Embeddings

This step verifies that the saved embeddings and their index can be reloaded correctly.

It checks the embedding dimensions, the number of records, the uniqueness of image IDs, and the presence of invalid numerical values.

In [9]:
import numpy as np
import pandas as pd

# Reload the saved files
saved_embeddings = np.load(EMBEDDINGS_PATH)
saved_index = pd.read_csv(INDEX_PATH)

# Verify the saved data
print("Embeddings shape:", saved_embeddings.shape)
print("Index records:", len(saved_index))
print("Unique image IDs:", saved_index["image_id"].nunique())
print("Missing values in index:", saved_index.isna().sum().sum())
print("NaN values in embeddings:", np.isnan(saved_embeddings).sum())
print("Infinite values in embeddings:", np.isinf(saved_embeddings).sum())

# Verify alignment between metadata and index
print(
    "Image IDs aligned:",
    metadata_df["image_id"].reset_index(drop=True).equals(
        saved_index["image_id"].reset_index(drop=True)
    )
)

# Verify embedding normalization
norms = np.linalg.norm(saved_embeddings, axis=1)
print("All embeddings normalized:", np.allclose(norms, 1.0, atol=1e-5))

Embeddings shape: (10099, 384)
Index records: 10099
Unique image IDs: 10099
Missing values in index: 0
NaN values in embeddings: 0
Infinite values in embeddings: 0
Image IDs aligned: True
All embeddings normalized: True


### Step 8: Inspect the Saved Embeddings

This step displays a small sample of the saved numerical embeddings to confirm that the file contains valid vector representations.

In [10]:
import numpy as np

# Load the saved embeddings
saved_embeddings = np.load(EMBEDDINGS_PATH)

# Display the first embedding
print("Embedding shape:", saved_embeddings.shape)
print("\nFirst 10 values of the first embedding:")
print(saved_embeddings[0][:10])

# Display the corresponding image ID
print("\nCorresponding image ID:")
print(embedding_index_df.iloc[0]["image_id"])

Embedding shape: (10099, 384)

First 10 values of the first embedding:
[-0.01507843  0.07417449  0.0561268   0.02540457  0.1162094  -0.00146455
  0.1318551  -0.04248518 -0.02420656  0.00747124]

Corresponding image ID:
36


## Step 9: Quality Control, Leakage Audit and Smoke Test

This step performs additional quality checks on the generated embeddings
and their corresponding index.

It verifies the consistency between the metadata, embeddings, and index files.

It also checks for potential data leakage between dataset partitions
and performs a basic smoke test to ensure that the embedding model
and saved embeddings can be used together correctly.

These checks help confirm that the generated embedding artifacts
are reliable and ready for the subsequent retrieval stage.

### 9.1 Load Saved Artifacts

This section reloads the previously generated embeddings and their
corresponding index from disk.

The final metadata dataset is also loaded to support additional
validation checks.

These files will be used to perform quality control, data leakage
auditing, and smoke testing.

### 9.1 Define File Paths and Load Saved Artifacts

This section defines the paths of the previously saved embedding
files and loads the generated embeddings, their index, and the
final metadata dataset.

These artifacts will be used for quality control, data leakage
auditing, and smoke testing.

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# Define project directories
PROJECT_DIR = Path(r"C:\Users\PC\Desktop\Metadata_RAG_START")
DATA_DIR = PROJECT_DIR / "data"
EMBEDDINGS_DIR = PROJECT_DIR / "embeddings"

# Define file paths
METADATA_PATH = DATA_DIR / "metadata.csv"
EMBEDDINGS_PATH = EMBEDDINGS_DIR / "image_embeddings.npy"
INDEX_PATH = EMBEDDINGS_DIR / "embedding_index.csv"

# Load saved artifacts
saved_embeddings = np.load(EMBEDDINGS_PATH)
saved_index = pd.read_csv(INDEX_PATH)

# Load final metadata
metadata_check = pd.read_csv(METADATA_PATH)

print("Artifacts loaded successfully!")
print("Embeddings shape:", saved_embeddings.shape)
print("Index shape:", saved_index.shape)
print("Metadata shape:", metadata_check.shape)

Artifacts loaded successfully!
Embeddings shape: (10099, 384)
Index shape: (10099, 4)
Metadata shape: (10099, 14)


### 9.2 Embedding Quality Control

This section performs additional validation checks on the saved
embeddings and their corresponding index.

It verifies the number of embeddings, vector dimensions, missing
values, invalid numerical values, image ID uniqueness, and
embedding normalization.

These checks help ensure the consistency and reliability of
the generated embedding artifacts.

In [7]:
print("Number of embeddings:", len(saved_embeddings))
print("Number of index records:", len(saved_index))
print("Embedding dimension:", saved_embeddings.shape[1])

print("Missing values in index:", saved_index.isna().sum().sum())
print("NaN values:", np.isnan(saved_embeddings).sum())
print("Infinite values:", np.isinf(saved_embeddings).sum())

print("Unique image IDs:", saved_index["image_id"].nunique())

norms = np.linalg.norm(saved_embeddings, axis=1)

print(
    "All embeddings normalized:",
    np.allclose(norms, 1.0, atol=1e-5)
)

print(
    "Index and embeddings aligned:",
    len(saved_index) == len(saved_embeddings)
)

Number of embeddings: 10099
Number of index records: 10099
Embedding dimension: 384
Missing values in index: 0
NaN values: 0
Infinite values: 0
Unique image IDs: 10099
All embeddings normalized: True
Index and embeddings aligned: True


### Results

All embedding quality control checks passed successfully.

The dataset contains 10,099 embeddings, each with 384 dimensions.
No missing values, NaN values, or infinite values were detected.

All image IDs are unique, the embedding index is aligned with
the saved embeddings, and all vectors are normalized.

The generated embedding artifacts are consistent and ready
for the next validation stage.

### 9.3 Data Leakage Audit

This section checks whether the same image ID or filename
appears in multiple dataset partitions.

It helps identify potential data leakage between training,
validation, and testing datasets.

In [8]:
print("Dataset partitions:")
print(metadata_check["dataset"].value_counts())

image_partition_counts = (
    metadata_check.groupby("image_id")["dataset"].nunique()
)

image_leakage = image_partition_counts[image_partition_counts > 1]

file_partition_counts = (
    metadata_check.groupby("file_name")["dataset"].nunique()
)

file_leakage = file_partition_counts[file_partition_counts > 1]

print("\nImage IDs in multiple partitions:", len(image_leakage))
print("Filenames in multiple partitions:", len(file_leakage))

if len(image_leakage) == 0 and len(file_leakage) == 0:
    print("\nNo cross-partition image overlap detected.")
else:
    print("\nPotential data leakage detected.")

Dataset partitions:
dataset
training      9999
validation     100
Name: count, dtype: int64

Image IDs in multiple partitions: 0
Filenames in multiple partitions: 0

No cross-partition image overlap detected.


### 9.4 Smoke Test

This section performs a basic retrieval test using a sample
text query and the saved image embeddings.

It verifies that query encoding and similarity-based retrieval
work correctly.

### 9.4.1 Load the Embedding Model

This section reloads the pre-trained Sentence Transformer model
used to generate the text embeddings.

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
Embedding dimension: 384


C:\Users\PC\AppData\Local\Temp\ipykernel_20176\2530536661.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


### 9.4.2 Perform the Smoke Test

A sample text query is encoded into an embedding and compared
with the saved image embeddings to retrieve the most similar image.

In [13]:
test_query = "A dog playing with a ball"

query_embedding = model.encode(
    [test_query],
    convert_to_numpy=True,
    normalize_embeddings=True
)[0]

print("Query embedding shape:", query_embedding.shape)

similarities = saved_embeddings @ query_embedding

best_index = np.argmax(similarities)
best_result = saved_index.iloc[best_index]

print("\nSmoke Test Result")
print("-----------------")
print("Query:", test_query)
print("Best embedding index:", best_index)
print("Image ID:", best_result["image_id"])
print("File name:", best_result["file_name"])
print("Similarity score:", similarities[best_index])

print("\nSmoke test completed successfully!")

Query embedding shape: (384,)

Smoke Test Result
-----------------
Query: A dog playing with a ball
Best embedding index: 238
Image ID: 12764
File name: 000000012764.jpg
Similarity score: 0.5212972

Smoke test completed successfully!


## Step 10 — Release Freeze (SHA-256 Manifest and Read-only)

This step prepares and protects the final project artifacts.

The metadata, image embeddings, and embedding index are copied
into a dedicated release directory.

SHA-256 hashes are generated and stored in a manifest file
to verify the integrity of the released files.

The files are then set to read-only to prevent accidental
modifications.

Finally, the release integrity and file permissions are verified
before proceeding to the dense retrieval demonstration.

### 10.1 Create the Release Directory

This section creates a dedicated directory for the final
release and copies the metadata, embeddings, and index files
into it.

In [14]:
import shutil
from pathlib import Path

# Define release directory
RELEASE_DIR = PROJECT_DIR / "release" / "v1"
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

# Define files to include in the release
release_files = [
    METADATA_PATH,
    EMBEDDINGS_PATH,
    INDEX_PATH
]

# Copy files into the release directory
for source_file in release_files:
    destination = RELEASE_DIR / source_file.name
    shutil.copy2(source_file, destination)
    print("Copied:", source_file.name)

print("\nRelease directory:", RELEASE_DIR)
print("Release files copied successfully!")

Copied: metadata.csv
Copied: image_embeddings.npy
Copied: embedding_index.csv

Release directory: C:\Users\PC\Desktop\Metadata_RAG_START\release\v1
Release files copied successfully!


### 10.2 Generate the SHA-256 Manifest

This section calculates the SHA-256 hash and file size
of each released artifact.

The results are stored in a manifest file for integrity
verification.

In [15]:
import hashlib
import json
from datetime import datetime, timezone

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


manifest = {
    "release_version": "v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimension": 384,
    "files": []
}

for file_path in release_files:
    released_file = RELEASE_DIR / file_path.name

    file_info = {
        "file_name": released_file.name,
        "size_bytes": released_file.stat().st_size,
        "sha256": calculate_sha256(released_file)
    }

    manifest["files"].append(file_info)

# Save manifest
MANIFEST_PATH = RELEASE_DIR / "manifest.json"

with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=4)

print("SHA-256 manifest generated successfully!")

for item in manifest["files"]:
    print("\nFile:", item["file_name"])
    print("Size:", item["size_bytes"], "bytes")
    print("SHA-256:", item["sha256"])

SHA-256 manifest generated successfully!

File: metadata.csv
Size: 15413522 bytes
SHA-256: f23c7155ac38f79dcaf54e143c74223c132d0d586804160bd7d06f3d7ddf0452

File: image_embeddings.npy
Size: 15512192 bytes
SHA-256: 538f2e08834276b2c7da3b41a48863f138559b3fcb5b4de59646da2a0481971a

File: embedding_index.csv
Size: 391125 bytes
SHA-256: 9eb42cb3c6b3090ed9a27baba2d317547782cb64b08e62864351f62a4d047ea2


### 10.3 Verify the Release Integrity

This section recalculates the SHA-256 hashes and compares
them with the values stored in the manifest.

It confirms whether the released files remain unchanged.

In [16]:
with open(MANIFEST_PATH, "r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

all_files_valid = True

for item in saved_manifest["files"]:
    file_path = RELEASE_DIR / item["file_name"]

    current_hash = calculate_sha256(file_path)
    expected_hash = item["sha256"]

    is_valid = current_hash == expected_hash

    print(item["file_name"], ":", "VALID" if is_valid else "MODIFIED")

    if not is_valid:
        all_files_valid = False

print("\nRelease integrity:", all_files_valid)

metadata.csv : VALID
image_embeddings.npy : VALID
embedding_index.csv : VALID

Release integrity: True


### 10.4 Set Release Files to Read-only

This section changes the file permissions to read-only
to prevent accidental modifications to the released artifacts.

In [17]:
import os
import stat

files_to_protect = [
    RELEASE_DIR / "metadata.csv",
    RELEASE_DIR / "image_embeddings.npy",
    RELEASE_DIR / "embedding_index.csv",
    MANIFEST_PATH
]

for file_path in files_to_protect:
    os.chmod(file_path, stat.S_IREAD)
    print("Set to read-only:", file_path.name)

print("\nRelease files are now read-only.")

Set to read-only: metadata.csv
Set to read-only: image_embeddings.npy
Set to read-only: embedding_index.csv
Set to read-only: manifest.json

Release files are now read-only.


### 10.5 Final Release Verification

This section verifies the integrity and read-only status
of all released files.

It confirms that the release is ready for the next stage.

In [18]:
# Verify file integrity again
integrity_check = all(
    calculate_sha256(RELEASE_DIR / item["file_name"]) == item["sha256"]
    for item in saved_manifest["files"]
)

# Verify read-only permissions
readonly_check = all(
    not (file_path.stat().st_mode & stat.S_IWRITE)
    for file_path in files_to_protect
)

print("Release integrity verified:", integrity_check)
print("Read-only status verified:", readonly_check)

if integrity_check and readonly_check:
    print("\nRelease v1 is ready!")
else:
    print("\nRelease verification requires attention.")

Release integrity verified: True
Read-only status verified: True

Release v1 is ready!


## Step 11 — Dense Retrieval Demo (G1 Naive RAG)

This step demonstrates the dense retrieval pipeline using the
frozen release artifacts.

A natural language query is converted into a normalized
embedding using the pre-trained Sentence Transformer model.

Cosine similarity is then calculated between the query embedding
and the saved image embeddings to retrieve the top-K most
similar images.

The retrieved images are matched with their corresponding
metadata, including captions and object categories, to build
a textual context for a future RAG generation stage.

This demonstration validates the basic text-to-image retrieval
workflow using semantic similarity.

### 11.1 Load the Frozen Release

This section loads the metadata, image embeddings, and index
from the frozen release directory.

These artifacts will be used for the dense retrieval demonstration.

In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

# Define the frozen release directory
PROJECT_DIR = Path(r"C:\Users\PC\Desktop\Metadata_RAG_START")
RELEASE_DIR = PROJECT_DIR / "release" / "v1"

# Load released artifacts
release_metadata = pd.read_csv(RELEASE_DIR / "metadata.csv")
release_embeddings = np.load(RELEASE_DIR / "image_embeddings.npy")
release_index = pd.read_csv(RELEASE_DIR / "embedding_index.csv")

print("Frozen release loaded successfully!")
print("Metadata shape:", release_metadata.shape)
print("Embeddings shape:", release_embeddings.shape)
print("Index shape:", release_index.shape)

Frozen release loaded successfully!
Metadata shape: (10099, 14)
Embeddings shape: (10099, 384)
Index shape: (10099, 4)


### 11.2 Load the Embedding Model

This section loads the same Sentence Transformer model
used during embedding preparation.

The model ensures that the user query is encoded into
the same vector space as the saved image embeddings.

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")
print("Embedding dimension:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
Embedding dimension: 384


### 11.3 Define the User Query

This section defines a natural language query that describes
the image the user wants to retrieve.

In [21]:
user_query = "A dog playing with a ball"

print("User query:", user_query)

User query: A dog playing with a ball


### 11.4 Encode the Query and Perform Dense Retrieval

The user query is converted into a normalized embedding.
Cosine similarity is calculated between the query embedding
and all saved image embeddings.

The top-K most similar images are then retrieved.

In [22]:
# Convert the query into an embedding
query_embedding = model.encode(
    [user_query],
    convert_to_numpy=True,
    normalize_embeddings=True
)[0]

# Check vector dimensions
assert query_embedding.shape[0] == release_embeddings.shape[1]

# Calculate cosine similarity
similarity_scores = release_embeddings @ query_embedding

# Retrieve the top 5 results
top_k = 5

top_indices = np.argsort(similarity_scores)[::-1][:top_k]

# Get the corresponding image records
top_results = release_index.iloc[top_indices].copy()

# Add similarity scores
top_results["similarity_score"] = similarity_scores[top_indices]

print("Query:", user_query)
print("\nTop", top_k, "Retrieved Images:")
display(top_results)

Query: A dog playing with a ball

Top 5 Retrieved Images:


,image_id,file_name,dataset,embedding_index,similarity_score
238,12764,000000012764.jpg,training,238,0.521297
5445,314953,000000314953.jpg,training,5445,0.518574
9362,543389,000000543389.jpg,training,9362,0.509609
3600,212276,000000212276.jpg,training,3600,0.488705
5822,337551,000000337551.jpg,training,5822,0.486868


### 11.5 Retrieve Metadata as Context

This section combines the retrieved image records with their
corresponding metadata.

Captions and object categories are extracted to create
a textual context for the retrieved results.

In [23]:
# Match retrieved images with their metadata
retrieved_context = top_results.merge(
    release_metadata[
        ["image_id", "file_name", "cleaned_captions", "object_categories"]
    ],
    on=["image_id", "file_name"],
    how="left"
)

# Display retrieved context
context_columns = [
    "image_id",
    "file_name",
    "similarity_score",
    "cleaned_captions",
    "object_categories"
]

display(retrieved_context[context_columns])

,image_id,file_name,similarity_score,cleaned_captions,object_categories
0,12764,000000012764.jpg,0.521297,"['A cat and a dog or playing with a ball.', 'c...","['cat', 'dog', 'sports ball']"
1,314953,000000314953.jpg,0.518574,['A black and white dog playing with a yellow ...,"['dog', 'frisbee']"
2,543389,000000543389.jpg,0.509609,"[""A brown dog holding a tennis ball init's mou...","['dog', 'person', 'sports ball']"
3,212276,000000212276.jpg,0.488705,['A DOG SPLASHING RUNNING IN WATER PLAYING FRI...,"['dog', 'frisbee']"
4,337551,000000337551.jpg,0.486868,['Dogs playing at a birthday with one dog wear...,"['dog', 'frisbee']"


### 11.6 Build the Retrieval Context

This section organizes the retrieved captions and object
categories into a textual context.

The context can be provided to a language model in a later
RAG generation stage.

In [24]:
context_parts = []

for _, row in retrieved_context.iterrows():
    item = (
        f"Image ID: {row['image_id']}\n"
        f"Caption: {row['cleaned_captions']}\n"
        f"Objects: {row['object_categories']}\n"
        f"Similarity: {row['similarity_score']:.4f}"
    )

    context_parts.append(item)

retrieval_context = "\n\n".join(context_parts)

print("Retrieved Context:")
print("------------------")
print(retrieval_context)

Retrieved Context:
------------------
Image ID: 12764
Caption: ['A cat and a dog or playing with a ball.', 'close up of a dog and a small cat with a white background', 'A brown dog and a black and white kitten playing with a tennis ball.', 'a brown and white dog and a cat playing with a ball', 'A kitten plays with a colorful ball with a dog next to it.']
Objects: ['cat', 'dog', 'sports ball']
Similarity: 0.5213

Image ID: 314953
Caption: ['A black and white dog playing with a yellow frisbee.', 'A small dog is racing towards a Frisbee.', 'A dog whizzes by to run towards a yellow Frisbee.', 'Action shot of a dog chasing a ball.', 'A dog chasing a frisbee in the grass.']
Objects: ['dog', 'frisbee']
Similarity: 0.5186

Image ID: 543389
Caption: ["A brown dog holding a tennis ball init's mouth.", 'A scruffy dog with a tennis ball in his mouth.', 'A dog with a tennis ball in its mouth looking out the window of a bus.', "A light colored dog with a tennis ball in it's mouth and a persons head 

### 11.7 Dense Retrieval Demonstration Conclusion

The dense retrieval pipeline was successfully demonstrated.

A natural language query was encoded into a normalized
embedding and compared with the saved image embeddings
using cosine similarity.

The top-K most similar images were retrieved, and their
captions and object categories were combined into a textual
context for a future RAG generation stage.

## General Conclusion

This phase covered embedding preparation, quality control,
release management, and dense retrieval.

A total of 10,099 normalized embeddings, each with 384 dimensions,
were generated using the `all-MiniLM-L6-v2` model.

The embeddings were saved in NumPy format in
`embeddings/image_embeddings.npy`. A separate index file,
`embeddings/embedding_index.csv`, was created to preserve
the association between each image and its corresponding embedding.

Quality control checks and a data leakage audit were performed.
The artifacts were then frozen in a dedicated release directory,
with SHA-256 hashes and read-only permissions.

Finally, a dense retrieval demonstration (G1 Naive RAG) was
performed using cosine similarity to retrieve the most relevant
images and build a textual context from their captions and
object categories.

This phase provides the validated embedding and retrieval
foundation for the next RAG generation stage.